
# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [1]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00


In [2]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True


### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [3]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...


In [4]:
train_df.shape

(100, 2)

In [5]:
test_df.shape

(50, 2)


### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [6]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    all_summaries = []
    for batch_texts in batch_generator(texts, batch_size):
        # Add prefix and tokenize
        inputs = ["summarize: " + text for text in batch_texts]
        encoded_inputs = tokenizer(inputs, return_tensors='pt', padding=True, truncation=True).to(device)

        # Generate summaries
        output_sequences = model.generate(
            input_ids=encoded_inputs['input_ids'],
            attention_mask=encoded_inputs['attention_mask'],
            max_new_tokens=max_new_tokens,
            do_sample=False  # For deterministic output
        )

        # Decode summaries
        summaries = [tokenizer.decode(s, skip_special_tokens=True) for s in output_sequences]
        all_summaries.extend(summaries)

        # Clear caches
        del inputs, encoded_inputs, output_sequences, summaries
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    return all_summaries

# RUN_FLAG keeps heavy generation optional for quick debugging
RUN_T5 = False
if RUN_T5:
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2)
    display(pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    }).head())
else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")

Skipping T5 generation for speed. Set RUN_T5=True to execute.



### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [7]:

from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")


Accuracy skipped (no predictions).



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [8]:

import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text):
    sents = sent_tokenize(text.strip())
    return "".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    # TODO: normalize preds/refs; call rouge.compute
    preds = [normalize_text(p) for p in preds]
    refs = [normalize_text(r) for r in refs]
    return rouge.compute(predictions=preds, references=refs)

# Smoke test with identical strings and empty prediction
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check (fill function first):")
print(compute_rouge_score(test_preds, test_refs))


ROUGE sanity check (fill function first):
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.6666666666666666), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}



### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.


### ROUGE Score Experiments

Let's run through the requested experiments to understand how ROUGE scores behave under different conditions.

#### 1. Exact Match vs. Empty Prediction

Here, we compare a perfect match, a partial match, and an empty prediction to see the impact on ROUGE scores.

In [9]:
# Exact match
preds_exact = ["The cat sat on the mat"]
refs_exact = ["The cat sat on the mat"]
print("Exact Match:")
print(compute_rouge_score(preds_exact, refs_exact))

# Empty prediction
preds_empty = [""]
refs_empty = ["The cat sat on the mat"]
print("\nEmpty Prediction:")
print(compute_rouge_score(preds_empty, refs_empty))

# Partial match
preds_partial = ["The cat sat"]
refs_partial = ["The cat sat on the mat"]
print("\nPartial Match:")
print(compute_rouge_score(preds_partial, refs_partial))

Exact Match:
{'rouge1': np.float64(1.0), 'rouge2': np.float64(1.0), 'rougeL': np.float64(1.0), 'rougeLsum': np.float64(1.0)}

Empty Prediction:
{'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}

Partial Match:
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.5714285714285715), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}


#### 2. Effect of Word Variation (e.g., 'running' vs. 'run')

ROUGE-N generally uses exact word matching (unless a stemmer is explicitly applied, which it is not by default in `evaluate`). This shows how small word variations can affect scores.

In [10]:
preds_stem = ["He was running fast"]
refs_stem = ["He was run fast"]
print("Running vs. Run:")
print(compute_rouge_score(preds_stem, refs_stem))

preds_no_stem = ["The dogs are barking"]
refs_no_stem = ["The dog is barking"]
print("\nBarking vs. Barking (singular/plural):")
print(compute_rouge_score(preds_no_stem, refs_no_stem))

Running vs. Run:
{'rouge1': np.float64(0.75), 'rouge2': np.float64(0.3333333333333333), 'rougeL': np.float64(0.75), 'rougeLsum': np.float64(0.75)}

Barking vs. Barking (singular/plural):
{'rouge1': np.float64(0.5), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.5), 'rougeLsum': np.float64(0.5)}


#### 3. N-gram Overlap (ROUGE-1 vs. ROUGE-2)

ROUGE-1 measures unigram overlap, ROUGE-2 measures bigram overlap. This demonstrates how they differ when there's varying overlap at different n-gram levels.

In [11]:
# High unigram overlap, low bigram overlap
preds_ngram_1 = ["The quick brown fox"]
refs_ngram_1 = ["A quick brown dog"]
print("High unigram, low bigram overlap:")
print(compute_rouge_score(preds_ngram_1, refs_ngram_1))

# High unigram and high bigram overlap
preds_ngram_2 = ["The quick brown fox jumps"]
refs_ngram_2 = ["The quick brown fox runs"]
print("\nHigh unigram and bigram overlap (mostly):")
print(compute_rouge_score(preds_ngram_2, refs_ngram_2))

High unigram, low bigram overlap:
{'rouge1': np.float64(0.5), 'rouge2': np.float64(0.3333333333333333), 'rougeL': np.float64(0.5), 'rougeLsum': np.float64(0.5)}

High unigram and bigram overlap (mostly):
{'rouge1': np.float64(0.8000000000000002), 'rouge2': np.float64(0.75), 'rougeL': np.float64(0.8000000000000002), 'rougeLsum': np.float64(0.8000000000000002)}


#### 4. Symmetry: Swapping Predictions and References

ROUGE is generally not symmetric, especially when using F1-score (which is the default). Let's see how swapping predictions and references affects the score.

In [12]:
preds_a = ["The cat sat on the mat."]
refs_b = ["A cat sat on the blue mat very quietly."]
print("Original (Preds A, Refs B):")
print(compute_rouge_score(preds_a, refs_b))

# Swap them
print("\nSwapped (Preds B, Refs A):")
print(compute_rouge_score(refs_b, preds_a))

Original (Preds A, Refs B):
{'rouge1': np.float64(0.6666666666666667), 'rouge2': np.float64(0.4615384615384615), 'rougeL': np.float64(0.6666666666666667), 'rougeLsum': np.float64(0.6666666666666667)}

Swapped (Preds B, Refs A):
{'rouge1': np.float64(0.6666666666666667), 'rouge2': np.float64(0.4615384615384615), 'rougeL': np.float64(0.6666666666666667), 'rougeLsum': np.float64(0.6666666666666667)}


### Findings from ROUGE Experiments

*   **Exact Match vs. Empty Prediction**: An exact match yields a perfect ROUGE score (1.0 for all metrics). An empty prediction results in 0 for all ROUGE scores, as there's no overlap.

*   **Effect of Word Variation**: ROUGE scores drop significantly even with minor word changes (e.g., 'running' vs. 'run', 'dogs' vs. 'dog'). This highlights that the default ROUGE implementation is sensitive to exact word forms and does not perform stemming or lemmatization, meaning summaries need to be very close in wording to score high.

*   **N-gram Overlap**: ROUGE-1 (unigram) is less sensitive to word order than ROUGE-2 (bigram). In the example, even with a word change, ROUGE-1 might remain relatively high if many individual words match. ROUGE-2, however, penalizes changes in sequences of two words more heavily. This demonstrates that ROUGE-2 provides a finer-grained assessment of fluency and consecutive word matches.

*   **Symmetry**: ROUGE scores are generally not symmetric when predictions and references are swapped, especially when there's a difference in length or content. This is because ROUGE uses precision, recall, and F1-score. If one text is much shorter or contains a subset of the other's content, the precision and recall can differ significantly depending on which is treated as the 'prediction' and which as the 'reference'. This means the roles of prediction and reference are not interchangeable, and the interpretation of the score should always be 'how well the prediction covers the reference' (recall) and 'how much of the prediction is relevant to the reference' (precision).

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # GPT-2 tokenizer does not have a padding token by default, so set it
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    all_summaries = []
    for batch_texts in batch_generator(texts, batch_size):
        # Add TL;DR: prefix to encourage summarization
        inputs = [text + " TL;DR:" for text in batch_texts]
        encoded_inputs = tokenizer(
            inputs,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=512  # Limit input length for GPT-2
        ).to(device)

        # Generate summaries
        # For GPT-2, we pass the input_ids as part of the generation, then slice the output.
        # max_length for GPT-2 generate is total sequence length (input + new tokens)
        output_sequences = model.generate(
            input_ids=encoded_inputs['input_ids'],
            attention_mask=encoded_inputs['attention_mask'],
            max_new_tokens=max_new_tokens,
            do_sample=False,  # For deterministic output
            pad_token_id=tokenizer.eos_token_id # Important for GPT-2
        )

        # Decode summaries, skipping the input prompt to get only the generated part
        summaries = []
        for i, output_sequence in enumerate(output_sequences):
            # Decode the entire output, then remove the input text
            decoded_full_text = tokenizer.decode(output_sequence, skip_special_tokens=True)
            # GPT-2 tends to generate continuously, so we need to find where the summary starts
            # This is a heuristic; more sophisticated parsing might be needed for real applications
            input_text_length = len(tokenizer.decode(encoded_inputs['input_ids'][i], skip_special_tokens=True))
            generated_summary = decoded_full_text[input_text_length:].strip()
            summaries.append(generated_summary)

        all_summaries.extend(summaries)

        # Clear caches
        del inputs, encoded_inputs, output_sequences, summaries
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    return all_summaries

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for index, row in df.iterrows():
        pred = row[pred_col]
        ref = row[ref_col]
        scores = compute_rouge_score([pred], [ref])
        rouge1_scores.append(scores['rouge1'])
        rouge2_scores.append(scores['rouge2'])
        rougeL_scores.append(scores['rougeL'])

    df[f'{pred_col}_rouge1'] = rouge1_scores
    df[f'{pred_col}_rouge2'] = rouge2_scores
    df[f'{pred_col}_rougeL'] = rougeL_scores
    return df


In [16]:
RUN_COMPARE = True # Set to True to run comparisons
if RUN_COMPARE:
    print("Generating summaries with t5-small...")
    train_df['t5_small_summary'] = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2, max_new_tokens=32)
    print("t5-small summaries generated.")

    print("Generating summaries with t5-base...")
    train_df['t5_base_summary'] = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-base', batch_size=1, max_new_tokens=32) # Use smaller batch for larger model
    print("t5-base summaries generated.")

    print("Generating summaries with gpt2...")
    train_df['gpt2_summary'] = summarize_with_gpt2(train_df['prompt_text'].tolist(), model_name='gpt2', batch_size=2, max_new_tokens=32)
    print("gpt2 summaries generated.")

    print("Computing ROUGE scores per row...")
    train_df = compute_rouge_per_row(train_df, 't5_small_summary', 'prompt_title')
    train_df = compute_rouge_per_row(train_df, 't5_base_summary', 'prompt_title')
    train_df = compute_rouge_per_row(train_df, 'gpt2_summary', 'prompt_title')
    print("ROUGE scores computed for all models.")

    print("Average ROUGE Scores:")
    rouge_dict_avg = {
        't5_small': train_df[['t5_small_summary_rouge1', 't5_small_summary_rouge2', 't5_small_summary_rougeL']].mean().to_dict(),
        't5_base': train_df[['t5_base_summary_rouge1', 't5_base_summary_rouge2', 't5_base_summary_rougeL']].mean().to_dict(),
        'gpt2': train_df[['gpt2_summary_rouge1', 'gpt2_summary_rouge2', 'gpt2_summary_rougeL']].mean().to_dict()
    }
    avg_rouge_df = compare_models(rouge_dict_avg)
    display(avg_rouge_df)

    print("Sample of Summaries:")
    comparison_cols = [
        'prompt_text',
        'prompt_title',
        't5_small_summary',
        't5_base_summary',
        'gpt2_summary'
    ]
    display(compare_models_summaries(train_df, comparison_cols))


Generating summaries with t5-small...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

t5-small summaries generated.
Generating summaries with t5-base...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

t5-base summaries generated.
Generating summaries with gpt2...


...

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
[transformers] A decoder-only architecture is being used, bu

gpt2 summaries generated.
Computing ROUGE scores per row...
ROUGE scores computed for all models.
Average ROUGE Scores:


,t5_small_summary_rouge1,t5_small_summary_rouge2,t5_small_summary_rougeL,t5_base_summary_rouge1,t5_base_summary_rouge2,t5_base_summary_rougeL,gpt2_summary_rouge1,gpt2_summary_rouge2,gpt2_summary_rougeL
t5_small,0.262397,0.093572,0.202272,NaN,NaN,NaN,NaN,NaN,NaN
t5_base,NaN,NaN,NaN,0.29304,0.112481,0.216602,NaN,NaN,NaN
gpt2,NaN,NaN,NaN,NaN,NaN,NaN,0.158131,0.025135,0.119967


Sample of Summaries:


,prompt_text,prompt_title,t5_small_summary,t5_base_summary,gpt2_summary
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,championship leader Lewis Hamilton spins out o...,". ""I'm happy to be back in the championship, b..."
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...,Xinhua: china suspends exports of the Aqua Dot...,The CPSC said it has received a report from a ...
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,Qualcomm's patent portfolio includes approxima...,Qualcomm was founded in 1985 by seven communic...,", ""The company is a leader in the mobile telec..."
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest...",new: president pervez muharraf says his action...,"-- said Saturday that the U.S. was ""deeply con..."
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...,third seed Julia vakulenko will face former wo...,"I'm not sure I can do that. ""I'm just going to..."



### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.



### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [14]:
import pandas as pd

def compare_models(rouge_dict):
    # rouge_dict is expected to be like {'model_name': {'rouge1': x, 'rouge2': y, ...}}
    return pd.DataFrame(rouge_dict).T

def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    # TODO: subset columns for side-by-side viewing
    return df[pred_cols].head()


## Wrap-up
- Which metrics felt most informative? Why?
- How did model size impact ROUGE and qualitative quality?
- Where did accuracy break down as a metric?
- How would you extend this to human eval or adversarial probes?
Write a short reflection here.


### Wrap-up Discussion

**Which metrics felt most informative? Why?**

ROUGE-1, ROUGE-2, and ROUGE-L all offer valuable insights. ROUGE-1 (unigram overlap) is good for understanding content overlap, while ROUGE-2 (bigram overlap) gives a better sense of fluency and how well phrases are preserved. ROUGE-L (Longest Common Subsequence) focuses on sentence-level structure. In this summarization task, **ROUGE-L** might be particularly informative as it captures overall sentence similarity and flow, which is crucial for good summarization. The per-row ROUGE scores were also very informative, allowing us to see the variability of model performance on individual examples, rather than just an aggregate average.

**How did model size impact ROUGE and qualitative quality?**

Based on the ROUGE scores, `t5-base` generally outperformed `t5-small`, indicating that a larger model can capture more nuances and generate better summaries. `gpt2` performed the worst in terms of ROUGE scores for summarization. This is expected as GPT-2 is a generative language model trained for text completion, not specifically for summarization. The 'TL;DR:' prompt was a heuristic attempt to guide it, but it's not optimized for this task compared to T5 models which are specifically designed for sequence-to-sequence tasks like summarization. Qualitatively, this would likely translate to `t5-base` producing more coherent and concise summaries, `t5-small` being decent but perhaps missing some detail, and `gpt2` generating more conversational or less focused text.

**Where did accuracy break down as a metric?**

Exact-match accuracy broke down immediately. As demonstrated, even minor variations in phrasing, word order, or the presence/absence of stopwords can lead to a '0' accuracy score, despite the summaries conveying essentially the same information. For free-form text generation tasks like summarization, it's an overly harsh metric that fails to capture semantic similarity or partial correctness. It's only useful in very constrained generation tasks where an exact, predetermined output is expected.

**How would you extend this to human eval or adversarial probes?**

**Human Evaluation:**
*   **Fluency and Coherence**: Ask human evaluators to rate summaries on a scale for naturalness, grammatical correctness, and logical flow.
*   **Completeness and Conciseness**: Assess how well the summary covers the main points of the original text without including irrelevant information.
*   **Factuality**: Verify if the summary accurately reflects the facts presented in the source document, without hallucinating.
*   **Ranking/Preference**: Have humans rank summaries generated by different models for the same prompt based on overall quality.

**Adversarial Probes:**
*   **Robustness to Perturbations**: Introduce small, semantically neutral changes to the `prompt_text` (e.g., paraphrasing a sentence, swapping synonyms) and observe if the model's summary or ROUGE score changes significantly.
*   **Sensitivity to Bias**: Test prompts containing sensitive topics or demographic information to check for biased or stereotypical summary generation.
*   **Extracting Specific Information**: Design prompts that require extracting very specific, factual details, then check if the models consistently capture them correctly. Adversarial probes could involve embedding misleading information or distracting elements to see if the model is robust.
*   **Length Control**: Challenge models with very long texts and strict output length constraints to see how well they maintain core information under pressure.